<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/notebook_13_pivot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 13.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

files = {
    "Naive": "results_naive.csv",
    "Seasonal Naive": "results_seasonal_naive.csv",
    "Linear Regression": "results_lr.csv",
    "Ridge": "results_regularized_lr.csv",
    "Lasso": "results_regularized_lr.csv",
    "ElasticNet": "results_regularized_lr.csv",
    "SARIMAX": "results_sarimax.csv",
    "Random Forest": "results_rf.csv",
    "XGBoost": "results_xgboost.csv",
    "LightGBM": "results_lgbm.csv",
    "CatBoost": "results_catboost.csv",
    "LSTM": "results_lstm.csv",
    "NBEATSx": "results_nbeats.csv",
    "NHiTS": "results_NHITS.csv",
    "TFT": "results_tft.csv",
}

In [ ]:
REG_MODELS = {'Ridge', 'Lasso', 'ElasticNet'}

dfs = []
loaded_reg = False
for model_name, filepath in files.items():
    tmp = pd.read_csv(filepath)
    tmp.columns = [c.lower() for c in tmp.columns]

    # для Ridge/Lasso/ElasticNet берём модель из файла
    if model_name in REG_MODELS:
        if loaded_reg:
            continue
        tmp = tmp[['модель', 'дом', 'горизонт', 'mae', 'rmse', 'mape']].copy()
        tmp.columns = ['модель', 'дом', 'горизонт', 'MAE', 'RMSE', 'MAPE']
        loaded_reg = True
    else:
        tmp = tmp[['дом', 'горизонт', 'mae', 'rmse', 'mape']].copy()
        tmp.columns = ['дом', 'горизонт', 'MAE', 'RMSE', 'MAPE']
        tmp['модель'] = model_name

    dfs.append(tmp)

df_all = pd.concat(dfs, ignore_index=True)

In [ ]:
model_order = [
    "Naive", "Seasonal Naive",
    "Linear Regression", "Ridge", "Lasso", "ElasticNet",
    "SARIMAX",
    "Random Forest", "XGBoost", "LightGBM", "CatBoost",
    "LSTM", "NBEATSx", "NHiTS", "TFT"
]

horizon_order = ["4h", "8h", "24h", "7d", "14d", "1m"]

# среднее MAPE по всем домам для каждой модели и горизонта
pivot_mape = df_all.groupby(["модель", "горизонт"])["MAPE"].mean().unstack()
pivot_mape = pivot_mape.reindex(model_order)[horizon_order]

print("Среднее MAPE по всем домам (%):")
print(pivot_mape.round(2).to_string())

pivot_mape.round(2).to_csv("results_summary_mape.csv")

Среднее MAPE по всем домам (%):
горизонт              4h     8h    24h     7d    14d     1m
модель                                                     
Naive              17.43  33.68  34.93  37.18  37.01  37.72
Seasonal Naive      7.43   7.22   7.30   7.65   7.75   8.40
Linear Regression  23.57  21.08   7.57   6.77   7.84  14.39
Ridge              23.88  21.27   7.61   6.69   7.67  14.19
Lasso              23.58  21.85   7.43   6.52   6.96  13.46
ElasticNet         24.00  21.91   7.64   6.68   7.18  13.49
SARIMAX            11.45  17.42  16.80  36.87  50.27  58.46
Random Forest       7.09   6.24   7.68   7.42   6.90  12.18
XGBoost             5.34   5.67   5.82   6.02   6.52   8.84
LightGBM            5.01   6.05   5.98   6.09   6.83   8.98
CatBoost            5.23   5.94   5.98   6.13   6.48   9.21
LSTM                5.65   5.91   6.50   7.49   7.96   8.11
NBEATSx             5.03   5.32   5.65   7.12   8.22  10.01
NHiTS               5.08   5.35   5.62   7.14   7.85   9.38
TFT     

In [ ]:
fig = go.Figure(go.Heatmap(
    z=pivot_mape.values,
    x=horizon_order,
    y=model_order,
    colorscale="RdYlGn_r",
    text=pivot_mape.round(1).values,
    texttemplate="%{text}%",
    textfont=dict(size=10),
    colorbar=dict(title="MAPE, %", thickness=15)
))

fig.update_layout(
    title="Среднее MAPE по всем домам (%)",
    xaxis_title="Горизонт прогноза",
    yaxis_title="Модель",
    width=700, height=450,
    font=dict(size=11),
    margin=dict(l=150, r=20, t=40, b=40)
)
fig.show()
fig.write_image("comparison_01_heatmap_mape.png", scale=2)

In [ ]:
model_colors = {
    "Naive": "#7f7f7f",
    "Seasonal Naive": "#bcbd22",
    "Linear Regression": "#17becf",
    "Ridge": "#aec7e8",
    "Lasso": "#1f77b4",
    "ElasticNet": "#6baed6",
    "SARIMAX": "#9467bd",
    "Random Forest": "#98df8a",
    "XGBoost": "#d62728",
    "LightGBM": "#ff7f0e",
    "CatBoost": "#e377c2",
    "LSTM": "#2ca02c",
    "NBEATSx": "#c5b0d5",
    "NHiTS": "#ffbb78",
    "TFT": "#8c564b",
}

fig2 = go.Figure()
for model in model_order:
    row = pivot_mape.loc[model]
    fig2.add_trace(go.Scatter(
        x=horizon_order,
        y=row.values,
        mode="lines+markers",
        name=model,
        line=dict(color=model_colors[model], width=2),
        marker=dict(size=6)
    ))

fig2.update_layout(
    title="MAPE по горизонтам прогноза (среднее по всем домам)",
    xaxis_title="Горизонт прогноза",
    yaxis_title="MAPE, %",
    width=700, height=450,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig2.show()
fig2.write_image("comparison_02_mape_by_horizon.png", scale=2)

In [ ]:
df_24h = df_all[df_all["горизонт"] == "24h"]

fig3 = go.Figure()
for model in model_order:
    row = df_24h[df_24h["модель"] == model].sort_values("дом")
    fig3.add_trace(go.Bar(
        x=row["дом"],
        y=row["MAPE"],
        name=model,
        marker_color=model_colors[model]
    ))

fig3.update_layout(
    title="MAPE по домам (горизонт 24ч)",
    xaxis_title="Дом",
    yaxis_title="MAPE, %",
    barmode="group",
    width=700, height=450,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig3.show()
fig3.write_image("comparison_03_mape_by_house_24h.png", scale=2)

In [ ]:
#Тест Диболда-Мариано для топ-5
import os

pred_files = {
    'LightGBM': 'predictions_lgbm.csv',
    'XGBoost':  'predictions_xgb.csv',
    'CatBoost': 'predictions_catboost.csv',
    'NBEATSx':  'predictions_nbeats.csv',
    'NHiTS':    'predictions_nhits.csv',
}

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

def diebold_mariano_test(e1, e2, h=1):
    """
    Тест Диболда-Мариано.
    e1 — ошибки базовой модели (LightGBM)
    e2 — ошибки сравниваемой модели
    h — горизонт прогноза
    H0: модели одинаково точны
    H1: e1 != e2
    Возвращает: DM-статистика, p-value
    """
    d = e1**2 - e2**2  # разность квадратичных потерь
    n = len(d)
    d_mean = np.mean(d)

    # автоковариационная дисперсия
    gamma = [np.mean((d - d_mean) * np.roll(d - d_mean, k))
             for k in range(h)]
    var_d = (gamma[0] + 2 * sum(gamma[1:])) / n

    if var_d <= 0:
        return np.nan, np.nan

    dm_stat = d_mean / np.sqrt(var_d)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return round(dm_stat, 4), round(p_value, 4)

# загружаем предсказания
pred_files = {
    'LightGBM': 'predictions_lgbm.csv',
    'XGBoost':  'predictions_xgb.csv',
    'CatBoost': 'predictions_catboost.csv',
    'NBEATSx':  'predictions_nbeats.csv',
    'NHiTS':    'predictions_nhits.csv',
}

preds = {}
for model, fname in pred_files.items():
    preds[model] = pd.read_csv(fname, parse_dates=['timestamp'])

# тест для каждого горизонта
horizons_list = ['4h', '8h', '24h', '7d', '14d', '1m']
competitors = ['XGBoost', 'CatBoost', 'NBEATSx', 'NHiTS']

dm_rows = []

for horizon_name in horizons_list:
    # ошибки LightGBM
    lgbm_h = preds['LightGBM'][preds['LightGBM']['horizon'] == horizon_name].copy()
    lgbm_h = lgbm_h.sort_values(['house_id', 'timestamp']).reset_index(drop=True)
    e_lgbm = lgbm_h['y_true'].values - lgbm_h['y_pred'].values

    for competitor in competitors:
        comp_h = preds[competitor][
            preds[competitor]['horizon'] == horizon_name
        ].copy()
        comp_h = comp_h.sort_values(['house_id', 'timestamp']).reset_index(drop=True)

        # выравниваем длины
        n = min(len(e_lgbm), len(comp_h))
        e_comp = comp_h['y_true'].values[:n] - comp_h['y_pred'].values[:n]
        e_base = e_lgbm[:n]

        dm_stat, p_val = diebold_mariano_test(e_base, e_comp)

        dm_rows.append({
            'Горизонт':     horizon_name,
            'Модель':       competitor,
            'DM-статистика':dm_stat,
            'p-value':      p_val,
            'Значимо (5%)': '+' if p_val is not np.nan and p_val < 0.05 else '-',
            'LightGBM лучше': '+' if dm_stat < 0 else '-'
        })

df_dm = pd.DataFrame(dm_rows)
df_dm.to_csv('results_dm_test.csv', index=False)

print('H0: модели одинаково точны | p < 0.05 → отвергаем H0')

for horizon_name in horizons_list:
    subset = df_dm[df_dm['Горизонт'] == horizon_name]
    print(f'\nГоризонт {horizon_name}:')
    print(f'{"Модель":<15} {"DM-стат":>10} {"p-value":>10} '
          f'{"Значимо":>10} {"LGBM лучше":>12}')
    print(f'  {"-"*57}')
    for _, row in subset.iterrows():
        print(f'{row["Модель"]:<15} {row["DM-статистика"]:>10.4f} '
              f'{row["p-value"]:>10.4f} {row["Значимо (5%)"]:>10} '
              f'{row["LightGBM лучше"]:>12}')

H0: модели одинаково точны | p < 0.05 → отвергаем H0

Горизонт 4h:
Модель             DM-стат    p-value    Значимо   LGBM лучше
  ---------------------------------------------------------
XGBoost            -0.6759     0.4991          -            +
CatBoost           -1.2546     0.2096          -            +
NBEATSx            -3.1839     0.0015          +            +
NHiTS              -3.1424     0.0017          +            +

Горизонт 8h:
Модель             DM-стат    p-value    Значимо   LGBM лучше
  ---------------------------------------------------------
XGBoost             0.9085     0.3636          -            -
CatBoost            1.5436     0.1227          -            -
NBEATSx            -3.2266     0.0013          +            +
NHiTS              -2.9371     0.0033          +            +

Горизонт 24h:
Модель             DM-стат    p-value    Значимо   LGBM лучше
  ---------------------------------------------------------
XGBoost            -0.8853     0.3760     

Тест Диболда-Мариано подтвердил что LightGBM статистически значимо превосходит нейросетевые модели NBEATSx и NHiTS на горизонтах 4h-14d (p < 0.05). Разница между LightGBM и XGBoost статистически незначима на большинстве горизонтов, что свидетельствует об эквивалентности этих моделей для данной задачи